# Notebook 07: UNet

**Module:** 07 Segmentation  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Understand encoder-decoder with skip connections
2. Implement UNet in PyTorch
3. Train on synthetic binary segmentation

## UNet (Ronneberger et al., 2015)

```
Encoder (contract)          Decoder (expand)
  Conv → Pool    ────────→  UpConv → Concat → Conv
  Conv → Pool    ──skip──→  UpConv → Concat → Conv
  ...                       ...
  Bottleneck
```

**Skip connections:** Preserve fine spatial details lost during downsampling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU())
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, features=(64,128,256)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        ch = in_ch
        for f in features:
            self.downs.append(DoubleConv(ch, f)); ch = f
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, 2, stride=2))
            self.ups.append(DoubleConv(f*2, f))
        self.final = nn.Conv2d(features[0], out_ch, 1)
    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i//2]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.ups[i+1](x)
        return self.final(x)

model = UNet(in_ch=3, out_ch=1)
print(f'UNet params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Output: {model(torch.randn(1,3,64,64)).shape}')

---

## Summary

UNet = encoder-decoder + skip connections. Foundation of GeoSpatial segmentation.

**Next:** [08_UNetPlusPlus.ipynb](08_UNetPlusPlus.ipynb)